In [1]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Varsayım: 'sequences' değişkeni bellekte yüklü bir numpy dizisidir.
# Şekli: (Örnek Sayısı, Zaman Uzunluğu) -> Örn: (1000, 2048)

class DASDataset(Dataset):
    def __init__(self, sequences, labels):
        # Veriyi PyTorch FloatTensor'a çeviriyoruz
        # TimesNet (Batch, Channel, Length) formatı bekler, bu yüzden 1 kanal ekliyoruz.
        self.data = torch.FloatTensor(sequences).unsqueeze(1) # Sonuç: (N, 1, T)
        self.labels = torch.LongTensor(labels)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

    def __len__(self):
        return len(self.data)

# Kullanım örneği:
# dataset = DASDataset(sequences, labels)
# dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [6]:
class TimesBlock(nn.Module):
    """
    TimesNet'in temel yapı taşı: 1D veriyi periyotlarına göre 2D'ye katlar ve işler.
    """
    def __init__(self, d_model, k=3):
        super(TimesBlock, self).__init__()
        self.k = k
        # 2D Inception Bloğu (Farklı ölçeklerdeki desenleri yakalamak için)
        self.conv = nn.Sequential(
            nn.Conv2d(d_model, d_model, kernel_size=1),
            nn.BatchNorm2d(d_model),
            nn.GELU(),
            nn.Conv2d(d_model, d_model, kernel_size=3, padding=1),
            nn.BatchNorm2d(d_model),
            nn.GELU()
        )

    def forward(self, x):
        B, C, T = x.shape
        # 1. FFT ile baskın frekansları bul
        fft_x = torch.fft.rfft(x, dim=-1)
        freq_amp = torch.abs(fft_x).mean(dim=0).mean(dim=0)
        # En güçlü 'k' frekansı (periyodu) seç
        top_k_indices = torch.topk(freq_amp, self.k).indices.detach().cpu().numpy()
        
        period_list =
        res =
        
        # 2. Her periyot için 1D -> 2D dönüşüm ve Konvolüsyon
        for period in period_list:
            # Padding (Bölünebilirlik için)
            pad_len = (period - T % period) % period
            x_pad = F.pad(x, (0, pad_len))
            
            # 2D Reshape: (Batch, Channel, Periyot Sayısı, Periyot Uzunluğu)
            x_2d = x_pad.reshape(B, C, -1, period)
            
            # 2D İşlem
            out_2d = self.conv(x_2d)
            
            # 1D'ye geri dönüş
            out_1d = out_2d.reshape(B, C, -1)
            res.append(out_1d) # Padding'i at
            
        # Farklı periyotlardan gelen özellikleri birleştir (Basit toplama veya ağırlıklı)
        return x + torch.stack(res).sum(dim=0)

class TimesNetGen(nn.Module):
    def __init__(self, seq_len, num_classes, d_model=32, latent_dim=64):
        super(TimesNetGen, self).__init__()
        
        # --- ENCODER ---
        self.embed = nn.Sequential(
            nn.Conv1d(1, d_model, kernel_size=3, padding=1), # Ham veriyi modele sok
            nn.BatchNorm1d(d_model),
            nn.GELU()
        )
        self.encoder_block = TimesBlock(d_model)
        
        # Latent Space (VAE Parametreleri)
        self.fc_mu = nn.Linear(d_model * seq_len, latent_dim)
        self.fc_var = nn.Linear(d_model * seq_len, latent_dim)

        # --- CONDITIONING ---
        self.label_emb = nn.Embedding(num_classes, 16) # Etiketi vektöre çevir

        # --- DECODER ---
        # Latent + Label -> Orijinal Boyut
        self.decoder_input = nn.Linear(latent_dim + 16, d_model * seq_len)
        self.decoder_block = TimesBlock(d_model)
        self.final_proj = nn.Conv1d(d_model, 1, kernel_size=3, padding=1)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x, labels):
        B, _, T = x.shape
        
        # 1. Encoding
        x_emb = self.embed(x)
        x_feat = self.encoder_block(x_emb)
        x_flat = x_feat.view(B, -1)
        
        mu = self.fc_mu(x_flat)
        logvar = self.fc_var(x_flat)
        z = self.reparameterize(mu, logvar)
        
        # 2. Conditioning (Etiketi Latent'a yapıştır)
        l_emb = self.label_emb(labels)
        z_cond = torch.cat([z, l_emb], dim=1)
        
        # 3. Decoding
        x_rec = self.decoder_input(z_cond).view(B, -1, T)
        x_rec = self.decoder_block(x_rec)
        out = self.final_proj(x_rec)
        
        return out, mu, logvar

SyntaxError: invalid syntax (2533908744.py, line 26)

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DASPhysicsLoss(nn.Module):
    def __init__(self, kld_weight=0.00025, spectral_weight=0.1):
        super(DASPhysicsLoss, self).__init__()
        self.kld_weight = kld_weight
        self.spectral_weight = spectral_weight
        self.mse = nn.MSELoss()

    def forward(self, recon_x, x, mu, logvar):
        # 1. Zaman Alanı Kaybı (Reconstruction Loss - Time Domain)
        # Sinyalin şeklini korur
        time_loss = self.mse(recon_x, x)

        # 2. Spektral Kayıp (Frequency Domain Loss)
        # Sinyalin frekans içeriğini (iş makinası vs. ayak sesi ayrımı) korur
        # FFT alıp genlik spektrumları arasındaki farka bakarız (sadece real kısım)
        fft_x = torch.fft.rfft(x, dim=-1).abs()
        fft_recon = torch.fft.rfft(recon_x, dim=-1).abs()
        spectral_loss = self.mse(fft_recon, fft_x)

        # 3. KL Divergence (Latent Space Regularization)
        # Gizli uzayın Gauss dağılımına uymasını sağlar (Üretim yapabilmek için şart)
        kld_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        # Batch boyutuna bölerek normalize edelim
        kld_loss = kld_loss / x.size(0)

        # Toplam Kayıp
        total_loss = time_loss + (self.spectral_weight * spectral_loss) + (self.kld_weight * kld_loss)
        
        return total_loss, time_loss, spectral_loss, kld_loss

In [4]:
class TimesNetGenConditional(nn.Module):
    def __init__(self, seq_len, num_classes, d_model=64, latent_dim=128):
        super(TimesNetGenConditional, self).__init__()
        
        # --- ENCODER ---
        self.encoder_embedding = nn.Linear(seq_len, d_model)
        # TimesBlock buraya gelecek (önceki rapordaki yapı)
        self.times_block = TimesBlock(d_model=d_model) 
        
        # Latent Space
        self.fc_mu = nn.Linear(d_model, latent_dim)
        self.fc_var = nn.Linear(d_model, latent_dim)

        # --- KOŞULLAMA (CONDITIONING) ---
        # Etiketi (label) bir vektöre çevirir
        self.label_embedding = nn.Embedding(num_classes, 16) 
        
        # --- DECODER ---
        # Latent vector + Label Embedding birleştirilir
        self.decoder_input = nn.Linear(latent_dim + 16, d_model)
        self.decoder_block = TimesBlock(d_model=d_model)
        self.final_projection = nn.Linear(d_model, seq_len)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x, labels):
        # 1. Encode
        x_enc = self.encoder_embedding(x) # (Batch, Channels, d_model)
        x_enc = self.times_block(x_enc)
        x_flat = x_enc.mean(dim=1)        # Pooling
        
        mu = self.fc_mu(x_flat)
        logvar = self.fc_var(x_flat)
        z = self.reparameterize(mu, logvar)
        
        # 2. Condition Injection (Etiketi Latent'a ekle)
        label_emb = self.label_embedding(labels) # (Batch, 16)
        z_cond = torch.cat([z, label_emb], dim=1) # (Batch, Latent+16)
        
        # 3. Decode
        x_dec = self.decoder_input(z_cond).unsqueeze(1)
        x_dec = self.decoder_block(x_dec)
        reconstructed = self.final_projection(x_dec)
        
        return reconstructed, mu, logvar

In [ ]:
import torch.optim as optim
from tqdm import tqdm

# Ayarlar
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_epochs = 50
learning_rate = 1e-3

# Model ve Loss Kurulumu
# seq_len: Verinizin zaman boyutu uzunluğu (örneğin 1000)
# num_classes: Kaç farklı sınıfınız varsa (kazı, yürüyüş, vb.)
model = TimesNetGenConditional(seq_len=sequences.shape[-1], num_classes=3).to(device)
criterion = DASPhysicsLoss(kld_weight=0.0005, spectral_weight=0.5).to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Dataloader (Önceki cevaptaki DASDataset sınıfını kullandığınızı varsayıyorum)
train_loader = DataLoader(DASDataset(sequences, labels, augment=True), batch_size=32, shuffle=True)

print("Eğitim Başlıyor...")

for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    for batch_idx, (data, targets) in enumerate(loop):
        # Veriyi GPU'ya at
        data = data.to(device).float()   # (Batch, 1, Seq_Len)
        targets = targets.to(device)     # (Batch)
        
        # 1. Forward Pass
        recon_batch, mu, logvar = model(data, targets)
        
        # 2. Loss Hesapla
        loss, loss_time, loss_fft, loss_kld = criterion(recon_batch, data, mu, logvar)
        
        # 3. Backward Pass
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient Clipping (Patlamayı önlemek için önemli)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        train_loss += loss.item()
        
        # İlerleme çubuğunda detayları göster
        loop.set_postfix({
            "Total": f"{loss.item():.4f}", 
            "Time": f"{loss_time.item():.4f}",
            "Freq": f"{loss_fft.item():.4f}",
            "KLD": f"{loss_kld.item():.4f}"
        })

    avg_loss = train_loss / len(train_loader)
    print(f"\tEpoch {epoch+1} Bitti. Ortalama Kayıp: {avg_loss:.5f}")

print("Eğitim Tamamlandı. Model sentetik veri üretimine hazır.")